# Comparison using Simulated Image Data

### Imports

In [1]:
import pprint

from torch.nn import MSELoss

from cocodeel.trainer import covar_trainer
from cocodeel.model import BaseNetwork, CovarNetwork
from cocodeel.posthoc_model import PostHocCovarNetwork
from cocodeel.benchmarking.posthoc_model import PostHocOrthNetwork

from experiments.simulation_images.backbone import TrafficBackbone
from experiments.simulation_images.utils import simulate_dataloader, evaluate_model

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


## Example

In [6]:
simulation_params = {
    'n': 1600, 'bz': 1., 'b2': 1., 'b3': 1.,
    'cv1': 0.5, 'cv2': 0.5, 'sdy': 1.
}
trainer_params = {
     'loss_fn': MSELoss(), 'epochs': 1000, 'lr': 1e-3, 'weight_decay': 1e-4, 'patience': 12
}
model_params = {
    'backbone': TrafficBackbone,
    'backbone_params': {"out_features": 32},
    'num_covariates': 1
}

In [7]:
base_models = []
covar_models = []
posthoc_models = []
posthoc_models_lam0 = []
posthoc_orth_models = []
posthoc_web_models = []

for i in range(3):
    train_loader, val_loader = simulate_dataloader(simulation_params, seed=i)

    # Train base model
    base_model = covar_trainer(
        model = BaseNetwork,
        model_params = model_params,
        train_loader = train_loader,
        val_loader = val_loader
    )
    base_model = base_model.center_effects(train_loader)
    base_models.append(base_model)

    posthoc_model = PostHocCovarNetwork(base_model, num_covariates=model_params['num_covariates'])
    posthoc_model = posthoc_model.fit(train_loader)
    posthoc_models.append(posthoc_model)

    posthoc_model_lam0 = PostHocCovarNetwork(base_model, num_covariates=model_params['num_covariates'])
    posthoc_model_lma0 = posthoc_model_lam0.fit(train_loader, lam=0.0)
    posthoc_models_lam0.append(posthoc_model_lam0)

    posthoc_orth_model = PostHocCovarNetwork(base_model, num_covariates=model_params['num_covariates'], orthogonalize=True)
    posthoc_orth_model = posthoc_orth_model.fit(train_loader)
    posthoc_orth_models.append(posthoc_orth_model)

    posthoc_web_model = PostHocOrthNetwork(base_model, num_covariates=model_params['num_covariates'])
    posthoc_web_model = posthoc_web_model.fit(train_loader)
    posthoc_web_models.append(posthoc_web_model)

    # Train covar model
    covar_model = covar_trainer(
        model = CovarNetwork,
        model_params = model_params,
        train_loader = train_loader,
        val_loader = val_loader
    )
    covar_model = covar_model.center_effects(train_loader)
    covar_models.append(covar_model)

In [8]:
test_simulaton_params = simulation_params.copy()
test_simulaton_params['n'] = 800

base_metrics = evaluate_model(base_models, test_simulaton_params)
covar_metrics = evaluate_model(covar_models, test_simulaton_params)
posthoc_metrics = evaluate_model(posthoc_models, test_simulaton_params)
posthoc_orth_metrics = evaluate_model(posthoc_orth_models, test_simulaton_params)
pho_metrics = evaluate_model(pho_models, test_simulaton_params)

In [9]:
print("Base Model Metrics")
pprint.pp(base_metrics)
print("Covariate Model Metrics:")
pprint.pp(covar_metrics)
print("Post-hoc Covariate Model Metrics:")
pprint.pp(posthoc_metrics)
print("Post-hoc Orthogonalized Covariate Model Metrics:")
pprint.pp(posthoc_orth_metrics)
print("Post-hoc Orthogonal Network Model Metrics:")
pprint.pp(pho_metrics)

Base Model Metrics
{'y': {'mspe': 1.0593176, 'bias2': 1.0555449, 'var': 0.0037727633},
 'fx': {'mspe': 0.064835526, 'bias2': 0.061999563, 'var': 0.0028359613},
 'fr': {'mspe': 0.13950257, 'bias2': 0.1366666, 'var': 0.0028359613},
 'fz': {'mspe': 0.07795459, 'bias2': 0.07795459, 'var': 0.0}}
Covariate Model Metrics:
{'y': {'mspe': 1.0711468, 'bias2': 1.0631238, 'var': 0.008023028},
 'fx': {'mspe': 0.07638143, 'bias2': 0.07095525, 'var': 0.00542618},
 'fr': {'mspe': 0.15500736, 'bias2': 0.14958118, 'var': 0.00542618},
 'fz': {'mspe': 0.09738242, 'bias2': 0.09539823, 'var': 0.0019841874}}
Post-hoc Covariate Model Metrics:
{'y': {'mspe': 1.0238706, 'bias2': 1.0144931, 'var': 0.00937761},
 'fx': {'mspe': 0.020111036, 'bias2': 0.00778674, 'var': 0.012324297},
 'fr': {'mspe': 0.057705395, 'bias2': 0.0453811, 'var': 0.012324297},
 'fz': {'mspe': 0.01864658, 'bias2': 0.008097426, 'var': 0.010549152}}
Post-hoc Orthogonalized Covariate Model Metrics:
{'y': {'mspe': 1.0205952, 'bias2': 1.0125965, 